Want to test why the GRPO performance first drop at the beginning of training. Is it because the required output format between prompt and RL reward model is different?

In [30]:
import pandas as pd

path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/data/gsm8k/train.parquet"
original_data = pd.read_parquet(path)
original_data_dict = original_data.to_dict(orient="records")


In [31]:
print(len(original_data_dict))

7473


In [ ]:
print(original_data_dict[0]["data_source"])

OK, so the prompt requires the model to put the answer after ####.

In [26]:
import gzip
import json
# path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/Llama3_GRPO_run1/rollouts/rollouts_step00000080_20251019T204514.jsonl.gz"
# path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/Llama3_GRPO_run1/rollouts/rollouts_step00000001_20251019T161719.jsonl.gz"
# path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/Llama3_GRPO_run2/rollouts/rollouts_step00000300_20251103T212404.jsonl.gz"
path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/Llama3_GRPO_run2/rollouts/rollouts_step00000001_20251103T050555.jsonl.gz"

RL_model_rollout_lines = []
with gzip.open(path, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        RL_model_rollout_lines.append(json.loads(line))
        # if i >= 10:
        #     break


In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/Llama3_GRPO_run1/global_step_20")

/root/miniconda3/envs/dsr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
print(RL_model_rollout_lines[0].keys())

dict_keys(['uid', 'index', 'prompt_length', 'response_length', 'prompt', 'response_token_ids', 'response_text', 'raw_chat', 'data_source', 'old_log_probs', 'ref_log_prob', 'token_level_scores', 'token_level_rewards', 'advantages', 'returns', 'input_ids', 'position_ids', 'sequence_score_sum', 'sequence_reward_sum', 'advantage_sum', 'return_sum'])


In [29]:
print(RL_model_rollout_lines[100]["response_text"])

<think>
Okay, let's see. Spencer jumps rope, and each time he jumps 4 times per minute. So, if he does a session for 10 minutes, how many jumps does he do in total? Hmm, that's straightforward. For each minute, he does 4 jumps, so 10 minutes would be 4 times 10, which is 40 jumps per session. Right, so each session is 40 jumps.

Then he does 2 sessions each day. So over 5 days, how many sessions is that? 5 times 2 is 10 sessions. So each day, he does 10 sessions? Wait, wait, no. Wait, the problem says he does 2 sessions each day. So if he does 2 sessions per day, then over 5 days, that's 2 multiplied by 5, which is 10 sessions. But each session is 10 minutes? Wait, hold on. Let me check again.

Wait, the problem says he does 2 sessions each day. So each day, he does 2 sessions. Each session is 10 minutes. So each session is 10 minutes, so per day, 2 sessions would be 2 times 10 minutes, which is 20 minutes. But maybe the problem is that each session is 10 minutes, and each session has 

In [24]:
print(tokenizer.decode(RL_model_rollout_lines[0]["prompt"], skip_special_tokens=True))


user

Two companies A and B, are selling bottled milk. Company A sells a big bottle for $4 and Company B sells a big bottle for $3.5. Company A was able to sell 300 and company B 350 big bottles of milk. How much more money did one company make from the other? Let's think step by step and output the final answer after "####".assistant




I don't understand. It seems that in thinking cot, the final answer is put after ####. But in final answer, the final answer is in \boxed. I don't why it's trained to put it in \boxed? 
And the reward model is indeed extracting the answer after ####? So strange... 

So indeed, at RL, the model is trained to output the final answer inboxed. It seems different to the adopted reward as well. Let me run one experiment to test which reward model is truly used. 

In [ ]:
from pprint import pprint

In [ ]:
df_dict = df.to_dict(orient="records")
print(df_dict[100].keys())
pprint(df_dict[100])

In [ ]:
import gzip
import json


In [ ]:
path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/deepscaler-1.5b-8k_run2/rollouts/rollouts_step00000001_20251019T161719.jsonl.gz"

with gzip.open(path, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        if i >= 10:
            break

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/deepscaler-1.5b-8k_run2/actor/global_step_80")

In [ ]:
tokenizer.decode(data["input_ids"])

In [ ]:
path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/data/Llama3_GSM8K_rollout/sharegpt.jsonl"

with open(path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

print(f"共读取 {len(data)} 条数据")
print("示例：")
print(json.dumps(data[0], indent=2, ensure_ascii=False))


In [ ]:
import json
path = "/cephfs/zhanghuaqing/RL/rllm/data/10.19_dpsk_gsm_rollout/gsm8k_train_sample00.jsonl"
with open(path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

In [ ]:

print(data[0])

In [ ]:
import json
from pprint import pprint
path = "/cephfs/zhanghuaqing/RL/rllm/data/10.19_dpsk_gsm_rollout/gsm8k_train_sharegpt_2rollouts.jsonl"
with open(path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]
pprint(data[0])


In [ ]:
import pandas as pd
from pathlib import Path

path = Path("/cephfs/zhanghuaqing/RL/rllm_deepscaler/checkpoints/deepscaler/deepscaler-1.5b-8k_run2/actor/global_step_20/boxed_test_result__shards/batch_000000.parquet")
if not path.exists():
    raise FileNotFoundError(f"Parquet file not found: {path}")

# Read parquet (requires pyarrow or fastparquet)
df = pd.read_parquet(path)

print(f"Loaded: {path}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}\n")
print("Columns and dtypes:")
print(df.dtypes)
print("\nFirst 10 rows:")
print(df.head(10))

In [ ]:
from pprint import pprint


In [ ]:
df_dict = df.to_dict(orient="records")
print(df_dict[1].keys())
pprint(df_dict[1])

In [ ]:
import json
from pprint import pprint
path = "/cephfs/zhanghuaqing/RL/rllm/data/10.19_dpsk_gsm_rollout/gsm8k_train_sharegpt_16rollouts_length4096.jsonl"
with open(path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]
pprint(data[22])


In [ ]:
import json
from pprint import pprint
path = "/cephfs/zhanghuaqing/RL/rllm_deepscaler/data/Llama3_GSM8K_rollout/sharegpt.jsonl"
with open(path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]
pprint(data[14])
